In [141]:
import pandas as pd
import numpy as np

In [142]:
### DEFINE BETA VALUES FOR VDF

beta_vals = {} # TBD: compute values based on growth

beta_template_aux = {
    "Period": ["AM", "MD", "PM", "NT"],
    1: [2.542, 2.545, 2.663, 2.545],
    2: [0.715, 0.708, 0.592, 0.708],
    3: [3.130, 3.133, 3.283, 3.286]
}

beta_vals_aux = pd.DataFrame(beta_template_aux)

beta_vals_aux.set_index("Period", inplace=True)

beta_vals[2025] = beta_vals_aux

In [143]:
### HERE ARE THE MEASURED SPEEDS WE USE AS REFERENCE

measured_speeds_file = r"inputs/measured_speeds.csv"

measured_speeds = pd.read_csv(
    measured_speeds_file,
    sep=",",          # `delimiter` y `sep` son equivalentes; elige uno
    encoding="utf-8",
    decimal=".",      # parsea decimales con punto
    thousands=",",    # parsea separador de miles con coma
    quotechar='"',
    index_col=0
)

measured_speeds

,1NB,2NB,3NB,4NB,5NB,6NB,7NB,1SB,2SB,3SB,4SB,5SB,6SB,7SB
Capacity Factors,,,,,,,,,,,,,,
Night,72.837666,70.283118,72.315008,71.584400,71.090572,72.258843,71.096500,73.519539,72.389295,70.356350,70.653439,73.088294,71.876692,72.957326
AM-Early,75.168898,71.369038,72.890479,71.950061,71.896036,72.565865,73.538587,71.655422,60.128947,54.332437,25.704033,45.227613,71.008205,76.128976
AM-Peak,73.894848,69.308665,70.884319,70.726660,71.691911,71.271828,73.327291,68.838611,45.473583,39.569322,21.512425,35.616745,62.701170,74.473008
AM-Shoulder,72.663268,68.124991,70.239916,70.128541,70.656450,71.443713,73.324812,69.380647,49.084855,43.836398,26.273427,46.608609,70.638371,75.903742
MD,71.399323,59.315348,62.108082,68.666765,71.233002,69.293181,73.197537,73.104724,57.133439,60.127074,56.744642,71.936138,72.223384,74.846889
PM-Shoulder,42.743099,32.939556,41.317598,58.317418,70.252699,49.856404,73.538587,74.084420,64.654782,62.725253,64.359712,73.122488,71.008205,74.797274
PM-Peak,28.933584,26.602488,37.248257,56.529138,58.932605,38.325598,67.084813,74.084420,70.605279,64.314453,60.995798,72.845997,70.639648,74.509868
PM-Late,63.630802,52.348926,59.338651,68.750423,71.205785,72.065832,72.553222,74.084420,72.446267,70.142102,70.566625,74.016071,72.269909,74.156643


In [144]:
### DEFINE LOOKUP TABLE FOR BONUS PER PERIOD

lookup_period_file = r"inputs/LookUp_Period.csv"

lookup_period = pd.read_csv(
    lookup_period_file,
    sep=",",          # `delimiter` y `sep` son equivalentes; elige uno
    encoding="utf-8",
    decimal=".",      # parsea decimales con punto
    thousands=",",    # parsea separador de miles con coma
    quotechar='"',
    index_col=0
)

# Clip y reasignar
lookup_period = lookup_period*0

lookup_period

,Bonus/Mile,4 Periods
Period,,
Night,0.0,
AM-Early,0.0,
AM-Peak,0.0,
AM-Shoulder,0.0,
MD,0.0,
PM-Shoulder,0.0,
PM-Peak,0.0,
PM-Late,0.0,


In [145]:
### DEFINE SEGMENT PARAMETERS
# Default configuration for time periods in traffic data

#TBD: Make this automatically
period_template = [                 # (Period, Hours/Day, Peak/OP, 4Periods tag)
    ("Night",        8, "OP",   "NT"),
    ("AM-Early",     1, "OP",   "AM"),
    ("AM-Peak",      2, "Peak", "AM"),
    ("AM-Shoulder",  1, "OP",   "AM"),
    ("MD",           5, "OP",   "MD"),
    ("PM-Shoulder",  1, "OP",   "PM"),
    ("PM-Peak",      3, "Peak", "PM"),
    ("PM-Late",      3, "OP",   "PM"),
]

rows = []
years = [2025]

# Default time periods list (for reference)
default_time_periods = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

# Create the base scenario: hour -> time period mapping
hour_to_period = {
    0: "Night",
    1: "Night",
    2: "Night",
    3: "Night",
    4: "Night",
    5: "Night",
    6: "AM-Early",
    7: "AM-Peak",
    8: "AM-Peak",
    9: "AM-Shoulder",
    10: "MD",
    11: "MD",
    12: "MD",
    13: "MD",
    14: "MD",
    15: "PM-Shoulder",
    16: "PM-Peak",
    17: "PM-Peak",
    18: "PM-Peak",
    19: "PM-Late",
    20: "PM-Late",
    21: "PM-Late",
    22: "Night",
    23: "Night"
}

period_to_period = {
    'Evening': 'Night',
    'Evening': 'PM-Late',
    'EarlyAM': 'AM-Early',
    'AM': 'AM-Peak',
    'AM': 'AM-Shoulder',
    'Midday': 'MD',
    'Midday': 'PM-Shoulder',
    'PM': 'PM-Peak'
}

# Define the segments and their parameters

peak_factor = 1 # 1.05 # Peak factor for adjustment at peak hour traffic

hov_percentage = pd.DataFrame({
    'Year' : [2025],
    'HOV percentage' : [0]
})

hov_percentage.set_index('Year', inplace=True)

lengths = [3,3,2.2,2.2,3.4,3.4,4.4,4.4,3,3,4.4,4.4,5.6,5.6]
inscope = [0.75]*10 + [0.65]*4

# Define segment parameters base
seg_params = pd.DataFrame({
    'SegDir':   ["1NB","1SB","2NB","2SB","3NB","3SB","4NB","4SB","5NB","5SB","6NB","6SB","7NB","7SB"],
    'Length':    lengths,
    'Inscope':   inscope,
    'Lanes_GP': [6.5,6.5,6.5,6.5,6.5,6.5,6.5,6.5,4.5,4.5,4.5,4.5,3.5,3.5], # We may need to sum the toll lane
    'Lanes_ML':  [1]*14, # Lanes_ML': [2,2,2,2,2,2,2,2,3,3,2,2,2,2], # Do test changing segment 5
    'CapPerLane_GP': [2000]*14,
    'CapPerLane_ML': [1800]*14, #[1800]*26,
    'Speed_GP':  [70]*2 + [70]*6 + [70]*6,
    'Speed_ML':  [75]*14,
    'Alpha_GP':  [1]*14,
    'Beta_GP':   [8]*14,
    'Alpha_ML':  [1]*14,
    'Beta_ML':   [4]*14,
    'Min_Toll_2016': [None]*14,
    'Max_Toll_2016': [None]*14,
    'LanesGP_AM_Peak': [5]*14,
    'LanesGP_PM_Peak': [5]*14,
})

seg_params.set_index('SegDir', inplace=True)

# Compute capacities as lanes * cap per lane
seg_params['Cap_GP'] = seg_params['Lanes_GP'] * seg_params['CapPerLane_GP']
seg_params['Cap_ML'] = seg_params['Lanes_ML'] * seg_params['CapPerLane_ML']

# Compute peak capacities as Alpha * base capacity
seg_params['CapGP_Peak'] = seg_params['Alpha_GP'] * seg_params['Cap_GP']
seg_params['CapML_Peak'] = seg_params['Alpha_ML'] * seg_params['Cap_ML']

# Optional: if you want integer capacities
seg_params[['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']] = seg_params[
    ['Cap_GP','Cap_ML','CapGP_Peak','CapML_Peak']
].astype(int)

# Preview
seg_params

,Length,Inscope,Lanes_GP,Lanes_ML,CapPerLane_GP,CapPerLane_ML,Speed_GP,Speed_ML,Alpha_GP,Beta_GP,Alpha_ML,Beta_ML,Min_Toll_2016,Max_Toll_2016,LanesGP_AM_Peak,LanesGP_PM_Peak,Cap_GP,Cap_ML,CapGP_Peak,CapML_Peak
SegDir,,,,,,,,,,,,,,,,,,,,
1NB,3.0,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
1SB,3.0,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
2NB,2.2,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
2SB,2.2,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
3NB,3.4,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
3SB,3.4,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
4NB,4.4,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
4SB,4.4,0.75,6.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,13000,1800,13000,1800
5NB,3.0,0.75,4.5,1,2000,1800,70,75,1,8,1,4,None,None,5,5,9000,1800,9000,1800


In [146]:
import numpy as np

def adjusted_cumprod(row, target_year, multiplier):
    years = row.index
    print(row.values)
    factors = 1 + row.values
    
    # Find the index of the target year
    target_idx = list(years).index(target_year)
    
    # Apply multiplier to the target year's factor
    factors[target_idx] *= multiplier
    
    # Calculate cumulative product
    return pd.Series(np.cumprod(factors), index=years)

In [147]:
### IMPORT GROWTHS FOR EACH CLASS
file_path_growths = r"inputs/growths_per_segment.csv"
base_growth_df = pd.read_csv(
    file_path_growths,
    delimiter=',',
    encoding='utf-8',
    decimal='.',        # ← this tells pandas how to parse decimals
    thousands=',',       # ← this tells pandas how to parse thousands
    quotechar='"'
)

base_growth_df = base_growth_df.iloc[:, 1:]
project_years = base_growth_df.columns[1:].tolist()
base_growth_df.iloc[:, 1:] =  base_growth_df.iloc[:, 1:] + 1

base_growth_df.loc[:, '2032'] *= 1.12

base_growth_df

,SegmentMapped,2025,2026,2027,2028,2029,2030,2031,2032,2033,...,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054
0,S1,1,1.014281,1.014281,1.014281,1.014281,1.016553,1.016553,1.138540,1.016553,...,1.014706,1.014706,1.014706,1.014706,1.014706,1.013360,1.013360,1.013360,1.013360,1.013360
1,S2,1,1.013951,1.013951,1.013951,1.013951,1.016255,1.016255,1.138206,1.016255,...,1.014505,1.014505,1.014505,1.014505,1.014505,1.013242,1.013242,1.013242,1.013242,1.013242
2,S3,1,1.014263,1.014263,1.014263,1.014263,1.016485,1.016485,1.138464,1.016485,...,1.014751,1.014751,1.014751,1.014751,1.014751,1.013492,1.013492,1.013492,1.013492,1.013492
3,S4,1,1.014321,1.014321,1.014321,1.014321,1.016514,1.016514,1.138496,1.016514,...,1.014982,1.014982,1.014982,1.014982,1.014982,1.013747,1.013747,1.013747,1.013747,1.013747
4,S5,1,1.014252,1.014252,1.014252,1.014252,1.016410,1.016410,1.138380,1.016410,...,1.015084,1.015084,1.015084,1.015084,1.015084,1.013887,1.013887,1.013887,1.013887,1.013887
5,S6,1,1.014025,1.014025,1.014025,1.014025,1.016204,1.016204,1.138149,1.016204,...,1.015290,1.015290,1.015290,1.015290,1.015290,1.014162,1.014162,1.014162,1.014162,1.014162
6,S7,1,1.013925,1.013925,1.013925,1.013925,1.016081,1.016081,1.138011,1.016081,...,1.015273,1.015273,1.015273,1.015273,1.015273,1.014151,1.014151,1.014151,1.014151,1.014151
7,S8,1,1.013881,1.013881,1.013881,1.013881,1.016023,1.016023,1.137946,1.016023,...,1.015348,1.015348,1.015348,1.015348,1.015348,1.014225,1.014225,1.014225,1.014225,1.014225
8,S9,1,1.013836,1.013836,1.013836,1.013836,1.015960,1.015960,1.137875,1.015960,...,1.015339,1.015339,1.015339,1.015339,1.015339,1.014214,1.014214,1.014214,1.014214,1.014214
9,S10,1,1.016112,1.016112,1.016112,1.016112,1.016112,1.018178,1.140359,1.018178,...,1.015975,1.015975,1.015975,1.015975,1.015975,1.014326,1.014326,1.014326,1.014326,1.014326


In [148]:
### IMPORT COUNTS AND SEPARATE BY CLASS AND PERIODS

file_path_counts = r"inputs/counts_by_hour_grouped_sorted.csv"
base_counts_df = pd.read_csv(
    file_path_counts,
    delimiter=',',
    encoding='utf-8',
    decimal='.',        # ← this tells pandas how to parse decimals
    thousands=',',       # ← this tells pandas how to parse thousands
    quotechar='"'
)

# --- Ajustar direcciones ---
base_counts_df["Direction"] = base_counts_df["Direction"].replace({"EB": "EB", "WB": "WB"})

# --- Crear columna Seg/Dir ---
base_counts_df["Seg/Dir"] = base_counts_df["Segment"].astype(str) + base_counts_df["Direction"]

# --- Función para procesar cada clase ---
def process_class(df_class):
    # Convertir a formato largo
    df_long = df_class.melt(
        id_vars=["Seg/Dir", "Segment", "Direction", "Class"],
        value_vars=[str(h) for h in range(24)],
        var_name="Hour",
        value_name="Volume"
    )
    
    # Mapear hora a periodo
    df_long["Hour"] = df_long["Hour"].astype(int)
    df_long["Period"] = df_long["Hour"].map(hour_to_period)
    
    # Agregar por Segment/Direction/Class/Period
    df_period = df_long.groupby(
        ["Seg/Dir", "Segment", "Direction", "Class", "Period"], as_index=False, sort=False
    ).agg({"Volume": "mean"}).round(0)
    
    # Pivot a formato ancho (periodos como columnas)
    period_order = df_period['Period'].unique()
    df_wide = df_period.pivot(
        index=["Seg/Dir", "Segment", "Direction", "Class"],
        columns="Period",
        values="Volume"
    )[period_order].reset_index()
    
    # Mantener solo Seg/Dir como índice
    df_proc = df_wide.drop(columns=["Class", "Direction", "Segment"]).set_index("Seg/Dir")
    
    return df_proc

# --- Separar por clases y procesar ---
dfs_by_class = {}
for cls in base_counts_df["Class"].unique():
    df_cls = base_counts_df[base_counts_df["Class"] == cls].copy()
    dfs_by_class[cls] = process_class(df_cls)


'''
Vehicle Classifications follow FHWA standards:
Lights: FHWA Classes 1-3 [Light Duty Vehicles]
Medium A: Classes 4-5 [Buses and Single Unit 2 axles trucks] 
Medium B: Class 6-7 [Single Unit 3 or 4 axles Trucks]
Heavy A: Classes 8-10 [Single Trailer 3 or more axles trucks]
Heavy B: Classes 11-13 [Combination Trucks Multitrailer Trucks]
'''

# --- Ejemplo de uso ---
df_lights = dfs_by_class["Lights"]
df_mediumA = dfs_by_class["Medium A"]
df_mediumB = dfs_by_class["Medium B"]
df_heavyA = dfs_by_class["Heavy A"]
df_heavyB = dfs_by_class["Heavy B"]

df_lights

Period,Night,AM-Early,AM-Peak,AM-Shoulder,MD,PM-Shoulder,PM-Peak,PM-Late
Seg/Dir,,,,,,,,
S1NB,2238.0,1841.0,4183.0,4812.0,5111.0,6757.0,6670.0,6393.0
S1SB,1637.0,4573.0,7757.0,7826.0,5968.0,4858.0,6246.0,4575.0
S2NB,3220.0,3086.0,6362.0,7274.0,7621.0,8273.0,7207.0,7436.0
S2SB,2585.0,6971.0,9309.0,8274.0,7694.0,5126.0,7001.0,6368.0
S3NB,3349.0,3209.0,6617.0,7565.0,7925.0,8604.0,7495.0,7734.0
S3SB,2688.0,7250.0,9681.0,8605.0,8002.0,5331.0,7281.0,6623.0
S4NB,3091.0,2962.0,6108.0,6983.0,7316.0,7941.0,6919.0,7138.0
S4SB,2482.0,6693.0,8936.0,7943.0,7386.0,4922.0,6721.0,6114.0
S5NB,1592.0,2313.0,4412.0,4147.0,4546.0,5076.0,5606.0,4570.0


In [149]:
# --- Lista de periodos según tus columnas ---
period_cols = ["Night","AM-Early","AM-Peak","AM-Shoulder","MD","PM-Shoulder","PM-Peak","PM-Late"]

# Diccionario de dataframes por clase
class_dfs = {
    "Lights": df_lights,
    "Medium A": df_mediumA,
    "Medium B": df_mediumB,
    "Heavy A": df_heavyA,
    "Heavy B": df_heavyB
}

projected_long_by_class = {}

for cls_name, df_class in class_dfs.items():
    df = df_class.copy()
    
    # Resetear índice Seg/Dir y extraer Segment y Direction
    df = df.reset_index()
    df["Segment"] = df["Seg/Dir"].str.extract(r"(\d+)")[0]    # solo los números
    df["Direction"] = df["Seg/Dir"].str.extract(r"([A-Z]+)")[0]  # solo las letras
    df["Class"] = cls_name
    
    # Melt usando las columnas de periodos
    df_long = df.melt(
        id_vars=["Seg/Dir","Segment","Direction","Class"],
        value_vars=period_cols,
        var_name="Period",
        value_name="AADT "+str(df["Class"][0])
    )
    
    # Normalizar SegDir (opcional)
    df_long["SegDir"] = df_long["Seg/Dir"].str.strip().str.upper().str.lstrip("S")
    
    projected_long_by_class[cls_name] = df_long

# Ejemplo: ver Lights
projected_long_lights_df = projected_long_by_class["Lights"]
projected_long_mediumA_df = projected_long_by_class["Medium A"]
projected_long_mediumB_df = projected_long_by_class["Medium B"]
projected_long_heaviesA_df = projected_long_by_class["Heavy A"]
projected_long_heaviesB_df = projected_long_by_class["Heavy B"]
projected_long_lights_df


,Seg/Dir,Segment,Direction,Class,Period,AADT Lights,SegDir
0,S1NB,1,S,Lights,Night,2238.0,1NB
1,S1SB,1,S,Lights,Night,1637.0,1SB
2,S2NB,2,S,Lights,Night,3220.0,2NB
3,S2SB,2,S,Lights,Night,2585.0,2SB
4,S3NB,3,S,Lights,Night,3349.0,3NB
...,...,...,...,...,...,...,...
107,S5SB,5,S,Lights,PM-Late,4118.0,5SB
108,S6NB,6,S,Lights,PM-Late,4463.0,6NB
109,S6SB,6,S,Lights,PM-Late,4021.0,6SB
110,S7NB,7,S,Lights,PM-Late,2896.0,7NB


In [150]:
rows = []

for year in years:
    for seg in seg_params.index:  # e.g., "1NB", "1SB", etc.
        seg_data = seg_params.loc[seg]
        # Extraer parte numérica y dirección
        seg_numeric = ''.join(filter(str.isdigit, seg))  # e.g., "10"
        direction = seg[len(seg_numeric):]       
        for p, hrs, peak, tag in period_template:
            rows.append({
                "Year": year,
                "SegDir": seg,
                "Segment": seg_numeric,        
                "Direction": direction,     
                "Period": p,
                "Hours/Day": hrs,
                "Peak": peak,
                "4Periods": tag,

                # Parámetros técnicos
                "Length": seg_data["Length"],
                "Speed GP": seg_data["Speed_GP"],
                "Capacity GP": seg_data["CapPerLane_GP"] * seg_data["Lanes_GP"],
                "Alpha GP": seg_data["Alpha_GP"],
                "Beta GP": seg_data["Beta_GP"],
                "Speed ML": seg_data["Speed_ML"],
                "Capacity ML": seg_data["CapPerLane_ML"] * seg_data["Lanes_ML"],
                "Alpha ML": seg_data["Alpha_ML"],
                "Beta ML": seg_data["Beta_ML"],
                "MinToll": 0.5,
                "MinCapture": 0
            })

# --- plantilla base ---
first_model_df = pd.DataFrame(rows)

# --- merge para todas las clases ---
for cls_name, df_proj in projected_long_by_class.items():
    proj_merge_df = df_proj[["SegDir", "Period", f"AADT {cls_name}"]].copy()
    proj_merge_df.rename(columns={"AADT": f"AADT {cls_name}"}, inplace=True)

    first_model_df = first_model_df.merge(
        proj_merge_df,
        on=["SegDir", "Period"],
        how="left"
    )

# --- 1. Reshape growths a formato largo ---
growths_long = base_growth_df.melt(
    id_vars="SegmentMapped",
    var_name="Year",
    value_name="AnnualGrowth"
).copy()
growths_long["Year"] = growths_long["Year"].astype(int)

# --- 2. Calcular crecimiento acumulado desde 2025 ---
# Ordenamos por año y aplicamos cumprod
growths_long = growths_long.sort_values(["SegmentMapped", "Year"])
growths_long["GrowthFactor"] = (growths_long["AnnualGrowth"]).groupby(growths_long["SegmentMapped"]).cumprod()

# Ahora GrowthFactor(y) = factor acumulado 2025→y

# --- 3. Preparar plantilla ---
fm = first_model_df.copy()
fm["Year"] = fm["Year"].astype(int)
fm["Segment"] = fm["Segment"].astype(str).str.replace(r"^S", "", regex=True)
fm["SegmentMapped"] = "S" + fm["Segment"].astype(str)

# --- 4. Merge GrowthFactor ---
fm = fm.merge(
    growths_long[["SegmentMapped", "Year", "GrowthFactor"]],
    on=["SegmentMapped", "Year"],
    how="left"
)

fm["GrowthFactor"] = fm["GrowthFactor"].fillna(1.0)

# --- 5. Aplicar GrowthFactor a todas las clases ---
for cls_name in projected_long_by_class.keys():
    col = f"AADT {cls_name}"
    if col in fm.columns:
        fm[col] = (fm[col].fillna(0) * fm["GrowthFactor"]).round(1)

# --- 6. Limpieza ---
fm = fm.drop(columns=["SegmentMapped"])   # opcional

first_model_df = fm

first_model_df


,Year,SegDir,Segment,Direction,Period,Hours/Day,Peak,4Periods,Length,Speed GP,...,Alpha ML,Beta ML,MinToll,MinCapture,AADT Lights,AADT Medium A,AADT Medium B,AADT Heavy A,AADT Heavy B,GrowthFactor
0,2025,1NB,1,NB,Night,8,OP,NT,3.0,70,...,1,4,0.5,0,2238.0,0.0,277.0,0.0,0.0,1.0
1,2025,1NB,1,NB,AM-Early,1,OP,AM,3.0,70,...,1,4,0.5,0,1841.0,0.0,228.0,0.0,0.0,1.0
2,2025,1NB,1,NB,AM-Peak,2,Peak,AM,3.0,70,...,1,4,0.5,0,4183.0,0.0,517.0,0.0,0.0,1.0
3,2025,1NB,1,NB,AM-Shoulder,1,OP,AM,3.0,70,...,1,4,0.5,0,4812.0,0.0,595.0,0.0,0.0,1.0
4,2025,1NB,1,NB,MD,5,OP,MD,3.0,70,...,1,4,0.5,0,5111.0,0.0,632.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,2025,7SB,7,SB,AM-Shoulder,1,OP,AM,5.6,70,...,1,4,0.5,0,3333.0,0.0,412.0,0.0,0.0,1.0
108,2025,7SB,7,SB,MD,5,OP,MD,5.6,70,...,1,4,0.5,0,2635.0,0.0,326.0,0.0,0.0,1.0
109,2025,7SB,7,SB,PM-Shoulder,1,OP,PM,5.6,70,...,1,4,0.5,0,2723.0,0.0,336.0,0.0,0.0,1.0
110,2025,7SB,7,SB,PM-Peak,3,Peak,PM,5.6,70,...,1,4,0.5,0,2613.0,0.0,323.0,0.0,0.0,1.0


In [151]:
first_model_df["Capacity GP"] = first_model_df.apply(
    lambda row: seg_params.loc[row["SegDir"], 'Cap_GP'],
    axis=1
)

first_model_df["B1"] = first_model_df.apply(
    lambda row: beta_vals[row["Year"]].loc[row["4Periods"], 1],
    axis=1
)

first_model_df["B2"] = first_model_df.apply(
    lambda row: beta_vals[row["Year"]].loc[row["4Periods"], 2],
    axis=1
)

# Here we load th value of the counts and we multiply the peak hour values by a constant
lights_w = 1

heavies_w = 3
heavies_w_toll = 3
heavies_w_vot = 4

medium_A_w = 3 # 1.5 # TBD: Maybe try 2.5 or 2.75 for every pce value
medium_A_w_toll = 4
medium_A_w_vot = 4

medium_B_w = 3 # 2.75
medium_B_w_toll = 5
medium_B_w_vot = 4

heavy_A_w = 3 # 2.75
heavy_A_w_toll = 3
heavy_A_w_vot = 3

heavy_B_w = 3
heavy_B_w_toll = 3
heavy_B_w_vot = 3

# 2. Compute TotalLights
first_model_df["TotalLights"] = first_model_df["AADT Lights"] 

first_model_df["TotalMediumA"] = first_model_df["AADT Medium A"]

first_model_df["TotalMediumB"] = first_model_df["AADT Medium B"]

first_model_df["TotalHeavyA"] = first_model_df["AADT Heavy A"]

first_model_df["TotalHeavyB"] = first_model_df["AADT Heavy B"]

first_model_df["TotalVeh"] = first_model_df.apply(
    lambda row: row["TotalLights"] + row["TotalMediumA"] + row["TotalMediumB"] + row["TotalHeavyA"] + row["TotalHeavyB"],
    axis=1
)

first_model_df["Corridor PCE pre-fix"] = first_model_df.apply(
    lambda row: row["TotalLights"] * lights_w + row["TotalMediumA"] * medium_A_w + row["TotalMediumB"] * medium_B_w + row["TotalHeavyA"] * heavy_A_w + row["TotalHeavyB"] * heavy_B_w,
    axis=1
)

first_model_df["Corridor PCE"] = first_model_df["Corridor PCE pre-fix"] # To anulate the suppression


first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalLights'] *= peak_factor

first_model_df["HOV3"] = first_model_df.apply(
    lambda row: row["TotalLights"] * hov_percentage.loc[row['Year']]['HOV percentage'],
    axis=1
)

first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalMediumA'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalMediumB'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalHeavyA'] *= peak_factor
first_model_df.loc[first_model_df['Period'].isin(['AM-Peak', 'PM-Peak']), 'TotalHeavyB'] *= peak_factor

first_model_df["TotalVeh"] = first_model_df.apply(
    lambda row: row["TotalLights"] + row["TotalMediumA"] + row["TotalMediumB"] + row["TotalHeavyA"] + row["TotalHeavyB"],
    axis=1
)

first_model_df["InScopeLights"] = first_model_df.apply(
    lambda row: (row["TotalLights"] - row["HOV3"]) * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeMediumA"] = first_model_df.apply(
    lambda row: row["TotalMediumA"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeMediumB"] = first_model_df.apply(
    lambda row: row["TotalMediumB"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeHeavyA"] = first_model_df.apply(
    lambda row: row["TotalHeavyA"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeHeavyB"] = first_model_df.apply(
    lambda row: row["TotalHeavyB"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

first_model_df["InScopeVeh"] = first_model_df.apply(
    lambda row: row["TotalVeh"] * seg_params.loc[row["SegDir"], 'Inscope'],
    axis=1
)

# first_model_df.to_csv('model_test.csv')

In [152]:
def get_speed(row):

    max_VC = 1.2  # TBD: check if we need to change this value
    ETC_discount = 0.15


    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speedGP =  row["Speed GP"] / (1 + row["Alpha GP"] * ((gp_pce / row["Capacity GP"]) ** row["Beta GP"]))

    timeGP = 60 * row["Length"] / speedGP

    gp_vc = gp_pce / row["Capacity GP"]

    return pd.Series([speedGP, timeGP], index=["Speed GP","Time GP"])

In [153]:
first_model_df[["Speed GP Real", "Time GP"]] = first_model_df.apply(
    get_speed, axis=1, result_type='expand'
)

first_model_df.to_csv('model_test.csv')

In [ ]:
def get_pce(row):

    measured_speed = measured_speeds.loc[row['Period']][row['SegDir']]

    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speed_rate = row['Speed GP'] / measured_speed

    if speed_rate < 1:

        measured_speed = row['Speed GP'] - 0.01

    gp_pce_aux = row["Capacity GP"] * ((row['Speed GP'] / (measured_speed * row['Alpha GP']) - 1 / row['Alpha GP']) ** (1 / row['Beta GP']))

    pce_factor = gp_pce_aux / gp_pce

    pce_factor = np.clip(pce_factor, a_min = 0.8, a_max = 1.2)

    return pce_factor

    # if (row["Period"] == "AM-Peak") or (row["Period"] == "PM-Peak") or (row["Period"] == "AM-Shoulder") or (row["Period"] == "AM-Early") or (row["Period"] == "PM-Shoulder") or (row["Period"] == "PM-Late"):
    #     return pce_factor
    # else:
    #     return 1

    

In [155]:
first_model_df["PCE Factor"] = first_model_df.apply(
    lambda row: get_pce(row),
    axis=1
)

# first_model_df.to_csv('model_test.csv')

In [156]:
cap_factor = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="PCE Factor"
)

cap_factor.to_csv('inputs/pce_factors.csv')

cap_factor

SegDir,1NB,1SB,2NB,2SB,3NB,3SB,4NB,4SB,5NB,5SB,6NB,6SB,7NB,7SB
Period,,,,,,,,,,,,,,
AM-Early,1.200000,0.8,1.016427,1.085255,0.976925,1.119791,1.058736,1.200000,0.938462,1.2,0.961196,0.8,1.200000,0.800000
AM-Peak,0.800000,0.8,0.838088,0.943216,0.800000,0.947875,0.800000,1.174886,0.800000,1.2,0.800000,1.2,1.127953,0.800000
AM-Shoulder,0.800000,0.8,0.832083,1.030160,0.800000,1.033129,0.800000,1.200000,0.800000,1.2,0.800000,0.8,0.954065,0.800000
MD,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.0,1.000000,1.000000
Night,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.0,1.000000,1.000000
PM-Late,1.112609,0.8,1.113331,0.800000,0.989384,0.800000,0.805124,0.800000,0.800000,0.8,0.800000,0.8,0.800000,0.808439
PM-Peak,1.200000,0.8,1.200000,0.800000,1.200000,0.961783,1.145756,1.110806,0.950189,0.8,1.170966,0.8,0.914877,0.800000
PM-Shoulder,1.200000,0.8,1.163504,1.200000,1.053184,1.200000,0.976681,1.200000,0.800000,0.8,1.182647,0.8,0.800000,0.800000


In [157]:
first_model_df["TotalLights"] = first_model_df.apply(
    lambda row: row["TotalLights"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalMediumA"] = first_model_df.apply(
    lambda row: row["TotalMediumA"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalMediumB"] = first_model_df.apply(
    lambda row: row["TotalMediumB"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalHeavyA"] = first_model_df.apply(
    lambda row: row["TotalHeavyA"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

first_model_df["TotalHeavyB"] = first_model_df.apply(
    lambda row: row["TotalHeavyB"] * cap_factor.loc[row["Period"], row["SegDir"]],
    axis=1
)

In [158]:
def get_capacity(row):

    measured_speed = measured_speeds.loc[row['Period']][row['SegDir']]

    gp_pce = (
        row["TotalLights"] * lights_w + 
        row["TotalMediumA"] * medium_A_w +
        row["TotalMediumB"] * medium_B_w +
        row["TotalHeavyA"] * heavy_A_w +
        row["TotalHeavyB"] * heavy_B_w
    )

    speed_rate = row['Speed GP'] / measured_speed

    if speed_rate < 1:

        measured_speed = row['Speed GP'] - 0.01

    capacity = gp_pce / ((row['Speed GP'] / (measured_speed * row['Alpha GP']) - 1 / row['Alpha GP']) ** (1 / row['Beta GP']))

    capacity_factor = capacity / row["Capacity GP"]

    return capacity_factor

In [159]:
first_model_df["Capacity Factor"] = first_model_df.apply(
    lambda row: get_capacity(row),
    axis=1
)

cap_factor = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="Capacity Factor"
)

cap_factor.to_csv('inputs/capacity_factors.csv')

cap_factor

SegDir,1NB,1SB,2NB,2SB,3NB,3SB,4NB,4SB,5NB,5SB,6NB,6SB,7NB,7SB
Period,,,,,,,,,,,,,,
AM-Early,0.704902,1.166555,1.000000,1.000000,1.000000,1.000000,1.000000,0.791136,1.000000,0.686135,1.000000,1.253014,0.661033,1.088760
AM-Peak,1.067171,1.090059,1.000000,1.000000,1.688229,1.000000,1.558322,1.000000,1.625612,0.783980,1.587170,0.997029,1.000000,1.627955
AM-Shoulder,1.227786,1.190659,1.000000,1.000000,1.929989,1.000000,1.781471,0.943132,1.527758,0.645280,1.491735,1.165645,1.000000,1.579220
MD,1.630115,1.903468,0.995633,0.977494,1.081638,1.057552,1.262557,0.934100,2.094182,2.058562,1.199533,2.010173,1.510006,1.560987
Night,0.713975,0.521814,1.026877,0.824014,1.068055,0.857049,0.985700,0.791677,0.733569,0.678122,0.715759,0.662329,0.455378,0.431615
PM-Late,1.000000,1.167485,1.000000,1.624578,1.000000,1.689904,1.000000,1.559997,1.684217,1.517543,1.644968,1.481788,1.372183,1.000000
PM-Peak,0.807734,1.593497,0.857881,1.785938,0.963650,1.000000,1.000000,1.000000,1.000000,1.826966,1.000000,1.783415,1.000000,1.238075
PM-Shoulder,0.904412,1.239139,1.000000,0.885929,1.000000,0.883059,1.000000,0.844206,1.870247,1.705992,1.000000,1.666205,1.547421,1.289575


In [160]:
period_order = [
    "Night",
    "AM-Early",
    "AM-Peak",
    "AM-Shoulder",
    "MD",
    "PM-Shoulder",
    "PM-Peak",
    "PM-Late"
]

first_model_df["Total Corridor"] = first_model_df.apply(
    lambda row: row["Corridor PCE"] * row["Hours/Day"],
    axis=1
)


first_model_df["Period"] = pd.Categorical(
    first_model_df["Period"],
    categories=period_order,
    ordered=True
)

corridor_pce = first_model_df.pivot(
    index=["Period"],
    columns="SegDir",
    values="Total Corridor"
)

# corridor_pce.to_csv('corridor_vals.csv')